# Calculate IAA

In [103]:
import json
from sklearn.metrics import cohen_kappa_score, jaccard_score, f1_score
from scipy.spatial.distance import dice
import os
import csv
from collections import defaultdict

In [77]:
dir_kim = '../resultsANON/KimIAABehav/'
dir_anna = '../resultsANON/AnnaIAABehav/'

files_kim = sorted(list(os.listdir(dir_kim)))
files_anna = sorted(list(os.listdir(dir_anna)))

files_kim = [f for f in files_kim if f.endswith('.csv')]
files_anna = [f for f in files_anna if f.endswith('.csv')]

In [78]:
for fk, fa in zip(files_kim, files_anna):
    print(fk)
    print(fa)
    print()
    break

output-13 Klassiek oud txt.1_LS-id3784.csv
output-13 Klassiek oud txt.1_LS-id512.csv



In [79]:
test_kim = files_kim[0]
test_anna = files_anna[0]
print(test_kim, test_anna)

output-13 Klassiek oud txt.1_LS-id3784.csv output-13 Klassiek oud txt.1_LS-id512.csv


In [80]:
def load_data(d, f, label):
    with open(f'{d}{f}') as infile:
        data = list(csv.DictReader(infile, delimiter = ','))
    labels = [d[label] for d in data]
    labels_split = []
    for l in labels:
        ls = l.split(' ')
        labels_split.extend(ls)
    return labels_split


def get_binary_labels(labels_kim, labels_anna):
    binary_k = []
    binary_a = []
    for n, (lk, la) in enumerate(zip(labels_kim, labels_anna)):
        #print(n, lk + ' -- ' + la)
        if lk != '' or la != '':
            if lk != '':
                binary_k.append(1)
            else:
                binary_k.append(0)
            if la != '':
                binary_a.append(1)
            else:
                binary_a.append(0)
    return binary_k, binary_a


def get_spans(labels_kim, labels_anna):
    k = defaultdict(set)
    a = defaultdict(set)
    
    lks = set()
    las = set()
    for n, (lk, la) in enumerate(zip(labels_kim, labels_anna)):
        if lk != '':
            lks.add(lk)
        if la != '':
            las.add(la)
    

    label_n_dict_k = dict()
    for n, l in enumerate(lks):
        label_n_dict_k[l] = n
    
    label_n_dict_a = dict()
    for n, l in enumerate(las):
        label_n_dict_a[l] = n

    for n, (lk, la) in enumerate(zip(labels_kim, labels_anna)):
        if lk != '' or la != '':
            if lk != '':
                nk = label_n_dict_k[lk]
                k[nk].add(n)
                
            if la != '':
                na = label_n_dict_a[la]
                a[na].add(n)
    
    union_labels = set(a.keys()).union(set(k.keys()))
    
    if len(k) > len(a):
        target = k
        comp = a
        target_name = 'kim'
        comp_name = 'anna'
    else:
        target = a
        comp = k
        target_name = 'anna'
        comp_name = 'kim'
    # print('most labels', target_name, len(k), len(a))
    
    
    overlap = defaultdict(list)
    comp_target = dict()
    for span, tokens in target.items():
        # get highest overlap with anna:
        ov_spans = defaultdict(list)
        for span_comp, tokens_comp in comp.items():
            ov = len(tokens.intersection(tokens_comp))
            tot = len(tokens.union(tokens_comp))
            per_ov = ov/tot
            ov_spans[per_ov].append(span_comp)
        max_per = max(ov_spans.keys())
        candidate_spans = ov_spans[max_per]
        # print(span, max_per, candidate_spans)
        for cs in candidate_spans:
            overlap[cs].append((max_per, span))
    for cs, ov_sp in overlap.items():
        if max(ov_sp)[0] == 0:
            pass
        else:
            comp_target[cs] = max(ov_sp)[1]
    # print()
    # print()
    spans_target = []
    spans_comp = []
    for span in union_labels:
        if span in target:
            tar = True
        else:
            tar = False
        if span in comp_target:
            comp = True
        else:
            comp = False
        # print(span, tar, comp)
        spans_target.append(tar)
        spans_comp.append(comp)
    if target_name == 'kim':
        return spans_target, spans_comp
    elif target_name == 'anna':
        return spans_comp, spans_target

In [107]:
all_spans_kim = []
all_spans_anna = []
label = 'Behav-predicate'
for fk, fa in zip(files_kim, files_anna):
    #print(fk, fa)
    labels_kim = load_data(dir_kim, fk, label)
    labels_anna = load_data(dir_anna, fa, label)
    labels_k, labels_a = get_spans(labels_kim, labels_anna)
    cks_f= cohen_kappa_score(labels_k, labels_a)
    #print('Per file kappa', cks_f)
    all_spans_kim.extend(labels_k)
    all_spans_anna.extend(labels_a)
    

print(len(all_spans_kim))
print(len(all_spans_anna))
print("Cohen's Kappa")
cks = cohen_kappa_score(all_spans_kim, all_spans_anna)
jac = jaccard_score(all_spans_kim, all_spans_anna)
print(cks)
print()
print("Jaccard score")
print(jac)
print()
print('Dice score')
di_score = dice(all_spans_kim, all_spans_anna)
print(di_score)
print()
print('f1')
f1_kim = f1_score(all_spans_kim, all_spans_anna)
f1_anna = f1_score(all_spans_anna, all_spans_kim) 
print(f1_kim, f1_anna)
print()
print('Percentage')
agree = 0
total = len(all_spans_kim)
for sp_k, sp_a in zip(all_spans_kim, all_spans_anna):
    # print(sp_k, sp_a)
    if sp_k == sp_a:
        agree += 1
percent_agree = agree/total
print(percent_agree)

1600
1600
Cohen's Kappa
-0.15654829760547662

Jaccard score
0.68375

Dice score
0.18782479584261322

f1
0.8121752041573868 0.8121752041573868

Percentage
0.68375


/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [105]:
all_binary_k = []
all_binary_a = []
label = 'Behav-predicate'
for fk, fa in zip(files_kim, files_anna):
    #print(fk, fa)
    labels_kim = load_data(dir_kim, fk, label)
    labels_anna = load_data(dir_anna, fa, label)
    binary_k, binary_a = get_binary_labels(labels_kim, labels_anna)
    cks_f = cohen_kappa_score(binary_k, binary_a)
    #print("per file kappa", cks_f)
    all_binary_k.extend(binary_k)
    all_binary_a.extend(binary_a)
print(len(all_binary_k), len(all_binary_a))
print("Cohen's Kappa tokens")
cks = cohen_kappa_score(all_binary_k, all_binary_a)
jac = jaccard_score(all_binary_k, all_binary_a)
print(cks)
print()
print("Jaccard score")
print(jac)
print()
print('Dice score')
di_score = dice(all_binary_k, all_binary_a)
print(di_score)
print()
print('f1')
f1_kim = f1_score(all_binary_k, all_binary_a)
f1_anna = f1_score(all_binary_a, all_binary_k) 
print(f1_kim, f1_anna)
print()
print('percentage')
agree = 0
total = len(all_binary_k)
for sp_k, sp_a in zip(all_binary_k, all_binary_a):
    # print(sp_k, sp_a)
    if sp_k == sp_a:
        agree += 1
percent_agree = agree/total
print(percent_agree)

9756 9756
Cohen's Kappa tokens
-0.06468415709009578

Jaccard score
0.7080770807708077

Dice score
0.17090734517522804

f1
0.8290926548247719 0.8290926548247719

percentage
0.7080770807708077


/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/piasommerauer/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:697: RuntimeWarnin

In [92]:
if not os.path.isdir('../IAA-data'):
    os.mkdir('../IAA-data')

In [93]:
# Write to same files for inspection

path_dir = '../IAA-data/'

label_main = 'Behav-predicate'
for fk, fa in zip(files_kim, files_anna):
    labels_kim = load_data(dir_kim, fk, label_main)
    labels_anna = load_data(dir_anna, fa, label_main)
    label = 'tok-id'
    token_ids = load_data(dir_kim, fk, label)
    label = 'token'
    tokens = load_data(dir_kim, fk, label)
    
    # make dicts
    token_dicts = []
    for tk_id, tok, k, a in zip(token_ids, tokens, labels_kim, labels_anna):
        tok_dict = dict()
        tok_dict['tok-id'] = tk_id
        tok_dict['token'] = tok
        tok_dict['Kim'] = k
        tok_dict['Anna'] = a
        token_dicts.append(tok_dict)
    
    header = token_dicts[0].keys()
    with open(f'{path_dir}{label_main}-{fk}', 'w') as outfile:
        writer = csv.DictWriter(outfile, fieldnames = header, delimiter = ',')
        writer.writeheader()
        for d in token_dicts:
            writer.writerow(d)

    
    
